# Content-Aware Image Retargeting
Reduce the width of images (`Baby.png`, `Diana.png`, `Snowman.png`) by `seams_number` columns using Seam Carving, implemented from scratch. Optionally use depth/saliency maps or other energy functions. Visualize seams if enabled.

In [75]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.animation import FuncAnimation

In [ ]:
def seam_carving(image, seams_number, depth_map=None, saliency_map=None, visualize=False):
    """
    Enhanced seam carving with depth and saliency integration.
    
    Args:
        image: Input image (H, W, 3) as float32 in [0, 1]
        seams_number: Number of seams to remove
        depth_map: Optional depth map (H, W) as float32
        saliency_map: Optional saliency map (H, W) as float32
        visualize: Whether to show seam removal animation
    
    Returns:
        resized_image: Image with seams removed
        seam_energies: List of energies for each removed seam
    """
    current_image = image.copy()
    current_depth = depth_map.copy() if depth_map is not None else None
    current_saliency = saliency_map.copy() if saliency_map is not None else None
    seam_energies = []
    
    animation_fig = None
    animation_ax = None
    
    print(f"Starting seam carving: removing {seams_number} seams")
    print(f"Original image size: {image.shape}")
    
    for i in range(seams_number):
        if len(current_image.shape) == 3:
            gray = cv2.cvtColor((current_image * 255).astype(np.uint8), cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        else:
            gray = current_image
        
        # Basic gradient energy
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        energy_map = np.sqrt(grad_x**2 + grad_y**2)
        
        # Add Canny edge information
        edges = cv2.Canny((gray * 255).astype(np.uint8), 50, 150).astype(np.float32) / 255.0
        energy_map += 0.5 * edges
        
        # Add depth map contribution
        if current_depth is not None and current_depth.shape == energy_map.shape:
            energy_map += 1.2 * current_depth
        
        # Add saliency map contribution
        if current_saliency is not None and current_saliency.shape == energy_map.shape:
            energy_map += 1.0 * current_saliency
        
        h, w = energy_map.shape
        dp = energy_map.copy()
        
        # Fill DP table
        for row in range(1, h):
            for col in range(w):
                if col == 0:
                    dp[row, col] += min(dp[row-1, col], dp[row-1, col+1])
                elif col == w-1:
                    dp[row, col] += min(dp[row-1, col-1], dp[row-1, col])
                else:
                    dp[row, col] += min(dp[row-1, col-1], dp[row-1, col], dp[row-1, col+1])
        
        seam = np.zeros(h, dtype=int)
        seam[-1] = np.argmin(dp[-1])
        
        for row in range(h-2, -1, -1):
            col = seam[row+1]
            if col == 0:
                seam[row] = col + np.argmin(dp[row, col:min(col+2, w)])
            elif col == w-1:
                seam[row] = max(0, col-1) + np.argmin(dp[row, max(0, col-1):col+1])
            else:
                seam[row] = col - 1 + np.argmin(dp[row, col-1:col+2])
        
        seam_energy = dp[-1, seam[-1]]
        seam_energies.append(seam_energy)
        
        if visualize:
            if animation_fig is None or animation_ax is None:
                plt.close('all')
                animation_fig = plt.figure('Seam Carving Animation', figsize=(12, 8))
                animation_ax = animation_fig.add_subplot(111)
            
            vis_img = current_image.copy()
            for row in range(h):
                if 0 <= seam[row] < vis_img.shape[1]:
                    vis_img[row, seam[row], :] = [1, 0, 0]
            
            animation_ax.clear()
            animation_ax.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
            animation_ax.set_title(f'Removing Seam #{i+1}', fontsize=16, fontweight='bold')
            animation_ax.axis('off')
            plt.draw()
            plt.pause(0.02)
        
        new_image = np.zeros((h, w-1, current_image.shape[2]), dtype=current_image.dtype)
        for row in range(h):
            col = seam[row]
            new_image[row, :, :] = np.delete(current_image[row, :, :], col, axis=0)
        current_image = new_image
        
        if current_depth is not None:
            new_depth = np.zeros((h, w-1), dtype=current_depth.dtype)
            for row in range(h):
                col = seam[row]
                new_depth[row, :] = np.delete(current_depth[row, :], col, axis=0)
            current_depth = new_depth
        
        if current_saliency is not None:
            new_saliency = np.zeros((h, w-1), dtype=current_saliency.dtype)
            for row in range(h):
                col = seam[row]
                new_saliency[row, :] = np.delete(current_saliency[row, :], col, axis=0)
            current_saliency = new_saliency

    
    print(f"Final image size: {current_image.shape}")
    if visualize and animation_fig is not None:
        plt.pause(1.0)
        plt.close(animation_fig)
        plt.ioff()
        plt.close('all')

    return current_image, seam_energies

In [77]:
def setup_seam_visualization():
    try:
        matplotlib.use('Qt5Agg')
        plt.ion()
    except:
        matplotlib.use('TkAgg')
        plt.ion()
setup_seam_visualization()

In [78]:
# Load images and maps
baby_path = './images/Baby/Baby.png'
baby_dmap_path = './images/Baby/Baby_DMap.png'
baby_smap_path = './images/Baby/Baby_SMap.png'

baby = cv2.imread(baby_path).astype(np.float32) / 255.0
baby_dmap = cv2.imread(baby_dmap_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
baby_smap = cv2.imread(baby_smap_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

# Parameters
seams_number = 200
visualize = True

In [79]:
# Process images
baby_resized, baby_energies = seam_carving(baby, seams_number, baby_dmap, baby_smap, visualize)

# Save seam energies
with open('seam_energy_log.txt', 'w') as f:
    f.write('Baby Seam Energies:\n' + '\n'.join(map(str, baby_energies)) + '\n')

# Visualize results
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(baby_resized, cv2.COLOR_BGR2RGB))
plt.title('Baby Resized')
plt.axis('off')
plt.show()

# Save resized images
cv2.imwrite('Baby_resized.png', (baby_resized * 255).astype(np.uint8))

Starting seam carving: removing 200 seams
Original image size: (370, 413, 3)
Final image size: (370, 213, 3)


True